# Heart Disease Prediction System

**Deployment Link:** _to be added after cloud deployment_

---


## Setup: Running This Notebook From Scratch

Follow the same steps as the Titanic notebook: create venv, install requirements, open Jupyter.


## Step 1: Import Libraries


In [ ]:
"""
Purpose of this code:
----------------------
Imports required libraries for building and evaluating the Heart Disease Prediction model.
"""
import numpy as np
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn import svm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from prettytable import PrettyTable
import matplotlib.pyplot as plt
import seaborn as sns


### Step 2: Load Sample Data


In [ ]:
sample_data = pd.read_csv("sample-data.csv")

print("\n\nSample Data:")
print("============\n")
pd.set_option("display.max_rows", None, "display.max_columns", None)
print(sample_data)


### Step 3.1: Understand Sample Data


In [ ]:
print("\n\nAttributes in Sample Data:")
print("==========================\n")
print(sample_data.columns)

print("\n\nNumber of Instances in Sample Data:", sample_data["ChestPain"].count())
print("========================================\n")


### Step 3.2: Pre-process Sample Data
- Sample Data is already Preprocessed
- No Preprocessing needs to be Performed


### Step 4: Feature Extraction
- Features are already Extracted
- No Feature Extraction needs to be Performed


### Step 5.1: Train the Label Encoder


In [ ]:
chestpain = pd.DataFrame({"ChestPain": ["Typical", "Atypical", "NonAnginal", "Asymptomatic"]})
sex = pd.DataFrame({"Sex": ["Male", "Female"]})
fbs = pd.DataFrame({"FBS": ["Normal", "High"]})
exang = pd.DataFrame({"ExAng": ["No", "Yes"]})
heartdisease = pd.DataFrame({"HeartDisease": ["Yes", "No"]})

chestpain_label_encoder = LabelEncoder()
sex_label_encoder = LabelEncoder()
fbs_label_encoder = LabelEncoder()
exang_label_encoder = LabelEncoder()
heartdisease_label_encoder = LabelEncoder()

chestpain_label_encoder.fit(np.ravel(chestpain))
sex_label_encoder.fit(np.ravel(sex))
fbs_label_encoder.fit(np.ravel(fbs))
exang_label_encoder.fit(np.ravel(exang))
heartdisease_label_encoder.fit(np.ravel(heartdisease))


### Step 5.2: Label Encode the Output


In [ ]:
sample_data_encoded_output = sample_data.copy()
original_sample_data = sample_data.copy()

print("\n\nHeartDisease Attribute After Label Encoding:")
print("===========================================\n")
sample_data["encoded_heartdisease"] = heartdisease_label_encoder.transform(sample_data['HeartDisease'])
print(sample_data[["HeartDisease", "encoded_heartdisease"]])

sample_data_encoded_output[['ChestPain', 'Sex', 'FBS', 'ExAng', 'HeartDisease']] = sample_data[['ChestPain', 'Sex', 'FBS', 'ExAng', 'encoded_heartdisease']]

sample_data_encoded_output.to_csv(r'sample-data-encoded-output.csv', index=False, header=True)
print("\nSaved: sample-data-encoded-output.csv")


### Step 5.3: Label Encode the Input


In [ ]:
sample_data_encoded = sample_data_encoded_output.copy()

sample_data_encoded_output["encoded_chestpain"] = chestpain_label_encoder.transform(sample_data_encoded_output['ChestPain'])
sample_data_encoded_output["encoded_sex"] = sex_label_encoder.transform(sample_data_encoded_output['Sex'])
sample_data_encoded_output["encoded_fbs"] = fbs_label_encoder.transform(sample_data_encoded_output['FBS'])
sample_data_encoded_output["encoded_exang"] = exang_label_encoder.transform(sample_data_encoded_output['ExAng'])

sample_data_encoded[['ChestPain', 'Sex', 'FBS', 'ExAng', 'HeartDisease']] = sample_data_encoded_output[['encoded_chestpain', 'encoded_sex', 'encoded_fbs', 'encoded_exang', 'HeartDisease']]

print("\n\nSample Data after Label Encoding:")
print("=================================\n")
print(sample_data_encoded)

sample_data_encoded.to_csv(r'sample-data-encoded.csv', index=False, header=True)


### Step 6.1: Splitting Sample Data into Training and Testing Data


In [ ]:
training_data_encoded, testing_data_encoded = train_test_split(
    sample_data_encoded,
    test_size=0.2,
    random_state=0,
    shuffle=False
)

training_data_encoded.to_csv(r'training-data-encoded.csv', index=False, header=True)
testing_data_encoded.to_csv(r'testing-data-encoded.csv', index=False, header=True)

print("Training Data shape:", training_data_encoded.shape)
print("Testing Data shape:", testing_data_encoded.shape)


### Step 6.2: Splitting Input Vectors and Outputs/Labels of Training Data


In [ ]:
input_vector_train = training_data_encoded.iloc[:, :-1]
output_label_train = training_data_encoded.iloc[:, -1]

print("Input vectors (train) shape:", input_vector_train.shape)
print("Output labels (train) shape:", output_label_train.shape)


### Step 6.3: Train the Support Vector Classifier


In [ ]:
print("\n\nTraining the Support Vector Classifier on Training Data")
print("========================================================\n")

svc_model = svm.SVC(gamma='auto', random_state=0)
svc_model.fit(input_vector_train, np.ravel(output_label_train))
print(svc_model)


### Step 6.4: Save the Trained Model


In [ ]:
pickle.dump(svc_model, open('svc_trained_model.pkl', 'wb'))
print("Saved: svc_trained_model.pkl")


### Step 7.1: Splitting Input Vectors and Outputs/Labels of Testing Data


In [ ]:
input_vector_test = testing_data_encoded.iloc[:, :-1]
output_label_test = testing_data_encoded.iloc[:, -1]

print("Input vectors (test) shape:", input_vector_test.shape)
print("Output labels (test) shape:", output_label_test.shape)


### Step 7.2: Load the Saved Model


In [ ]:
model = pickle.load(open('svc_trained_model.pkl', 'rb'))
print("Model loaded from svc_trained_model.pkl")


### Step 7.3: Evaluate the Machine Learning Model


In [ ]:
model_predictions = model.predict(input_vector_test)

pd.options.mode.chained_assignment = None
testing_data_encoded["Predictions"] = model_predictions
testing_data_encoded.to_csv(r'model-predictions.csv', index=False, header=True)

model_predictions = testing_data_encoded
print("\n\nPredictions Returned by svc_trained_model:")
print("==========================================\n")
print(model_predictions)


### Step 7.4: Calculate the Accuracy Score


In [ ]:
model_accuracy_score = accuracy_score(
    model_predictions["HeartDisease"],
    model_predictions["Predictions"]
)

print("\n\nAccuracy Score:")
print("===============\n")
print(round(model_accuracy_score, 2))


### Step 7.5: Detailed Evaluation Metrics


In [ ]:
print("\nClassification Report:")
print("=======================\n")
print(classification_report(
    model_predictions["HeartDisease"],
    model_predictions["Predictions"],
    target_names=["No Disease (0)", "Disease (1)"]
))

conf_matrix = confusion_matrix(model_predictions["HeartDisease"], model_predictions["Predictions"])
conf_matrix_df = pd.DataFrame(
    conf_matrix,
    index=["Actual: No Disease", "Actual: Disease"],
    columns=["Predicted: No Disease", "Predicted: Disease"]
)
print("\nConfusion Matrix:")
print("==================\n")
print(conf_matrix_df)

plt.figure(figsize=(5, 4))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - SVC on Heart Disease Testing Data')
plt.tight_layout()
plt.show()


### Step 8.1: Take Input from User


In [ ]:
# Demo patient (edit these values, or swap in input() calls)
chestpain_input = "Typical"
sex_input = "Male"
fbs_input = "Normal"
exang_input = "No"

print(f"ChestPain: {chestpain_input}, Sex: {sex_input}, FBS: {fbs_input}, ExAng: {exang_input}")


### Step 8.2: Convert User Input into Feature Vector


In [ ]:
user_input = pd.DataFrame({
    'ChestPain': [chestpain_input],
    'Sex': [sex_input],
    'FBS': [fbs_input],
    'ExAng': [exang_input]
})

print("\n\nUser Input Feature Vector:")
print("==========================\n")
print(user_input)


### Step 8.3: Label Encoding of Feature Vector


In [ ]:
unseen_data_features = user_input.copy()

unseen_data_features["ChestPain"] = chestpain_label_encoder.transform(user_input['ChestPain'])
unseen_data_features["Sex"] = sex_label_encoder.transform(user_input['Sex'])
unseen_data_features["FBS"] = fbs_label_encoder.transform(user_input['FBS'])
unseen_data_features["ExAng"] = exang_label_encoder.transform(user_input['ExAng'])

print("\n\nUser Input Encoded Feature Vector:")
print("==================================\n")
print(unseen_data_features)


### Step 8.4: Load the Saved Model


In [ ]:
model = pickle.load(open('svc_trained_model.pkl', 'rb'))


### Step 8.5: Model Prediction


In [ ]:
predicted = model.predict(unseen_data_features)

if predicted == 1:
    prediction = "HEART DISEASE (Yes)"
if predicted == 0:
    prediction = "NO HEART DISEASE (No)"

pretty_table = PrettyTable()
pretty_table.add_column("       ** Prediction **       ", [prediction])

print(pretty_table)


### Step 9: Execute the Feedback Phase
#### Step 9.1: Collect Feedback


In [ ]:
example_feedback = pd.DataFrame({
    "Reviewer": ["Domain Expert (Cardiology)", "Student Tester", "Student Tester"],
    "Case": [
        "Typical chest pain, male, normal FBS, no exang -> predicted NO DISEASE",
        "Asymptomatic, male, high FBS, exang yes -> predicted DISEASE",
        "Atypical chest pain, female -> predicted NO DISEASE (unexpected for some cases)",
    ],
    "Feedback": [
        "Matches clinical expectation: typical angina with no risk factors often benign.",
        "Matches clinical expectation: asymptomatic + risk factors indicates disease.",
        "Model may miss atypical presentations in women; needs more data.",
    ],
})
print(example_feedback.to_string(index=False))


#### Step 9.2: Potential Improvements


In [ ]:
improvements = [
    "Use the full UCI Heart Disease dataset (all 4 databases) instead of Cleveland only",
    "Add numeric features such as Age, Cholesterol, Resting BP, Max Heart Rate",
    "Try class-balanced evaluation and cross-validation",
    "Compare additional models (Random Forest, Gradient Boosting, Logistic Regression)",
]
for i, item in enumerate(improvements, 1):
    print(f"{i}. {item}")


#### Step 9.3: Improve the Model Based on Feedback


In [ ]:
print("Next version: retrain with Age + Cholesterol + RestingBP features, then re-evaluate before replacing svc_trained_model.pkl")


## Step 10: Deploy Your Model as a Public Web App

Follow the same deployment process as the Titanic notebook (PythonAnywhere / Streamlit Cloud).
Upload the notebook, sample-data.csv, and svc_trained_model.pkl. Create a web app that loads the model and encoders.

**Deployment Link:** _to be added_


## Conclusion

The Heart Disease Prediction System uses four input attributes (Chest Pain Type, Sex, Fasting Blood Sugar, Exercise-Induced Angina) to predict presence of heart disease. The SVC model is trained and evaluated on the Cleveland dataset.
